# Shared Configuration Notebook

This notebook sets up the entire EO-Series pipeline. Every main notebook (`01.`, `02.`, `03.`, `04.`) runs `%run ./config_nb` to establish catalog / schema / Volume paths, install GeoBrix, instantiate the `StacClient`, and register shared helper functions.

__Libraries__

* GeoBrix is installed below (the lightweight `[light]` wheel) — nothing is assumed pre-staged on the cluster.
* Default tier is **lightweight** (`databricks.labs.gbx.pyrx`), which runs on **Serverless**. Flip to option-2 (`rasterx`) for the heavyweight tier on a classic x86 cluster (JAR + GDAL init script).
* Pandas 2.2.3 with DBR 17.3.

__Unity Catalog__

* Replace `catalog_name` and `schema_name` with your preferred locations.
* Volume 'data' must exist under `catalog_name`/`schema_name`.

In [0]:
# -- GeoBrix: lightweight tier (option-1, default). Installed here so nothing is
#    assumed pre-staged on the cluster. For the heavyweight tier (option-2 below)
#    attach the GeoBrix JAR + GDAL init script to a classic x86 cluster instead.
%pip install --quiet --disable-pip-version-check --ignore-installed --no-deps geobrix
%pip install --quiet "geobrix[light,stac] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.4.0-py3-none-any.whl"

# -- EO-series viz extras; shapely / rasterio / pyogrio / pystac-client / planetary-computer / tenacity come via geobrix[light,stac]
%pip install --quiet folium mapclassify geopandas rich

In [0]:
# -- import databricks + delta + spark functions
from delta.tables import *
from pyspark.databricks.sql import functions as DBF
from pyspark.sql import functions as F
from pyspark.sql.functions import col, udf, pandas_udf
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# -- other imports
from datetime import datetime
from io import BytesIO
from matplotlib import pyplot
from rasterio.io import MemoryFile

from databricks.labs.gbx.stac import StacClient

import geopandas as gpd
import os
import pandas as pd
import pathlib
import pyspark
import requests
import warnings

date_format = "%Y-%m-%d %H:%M:%S"
warnings.simplefilter("ignore")

## STAC Client Setup

`StacClient` provides distributed STAC catalog searches and signed asset downloads on Planetary Computer. The default instance connects to Planetary Computer with automatic request signing. See [STAC API](https://databrickslabs.github.io/geobrix/docs/api/stac).

In [0]:
# -- StacClient: Planetary Computer (default catalog), auto-signed requests
stac_client = StacClient()   # default catalog = Planetary Computer, sign = planetary_computer

In [0]:
# -- Spark conf tuning, guarded for Serverless --
# Serverless forbids runtime spark.conf.set; set_conf_safe() no-ops there (Adaptive
# Query Execution handles partitioning). On classic clusters it applies the explicit
# high-parallelism tuning the download / stacking steps benefit from.
def set_conf_safe(key, value):
  try:
    spark.conf.set(key, value)
    return True
  except Exception as e:
    print(f"... skipping spark.conf.set({key}) [Serverless?]: {type(e).__name__}")
    return False

set_conf_safe("spark.sql.adaptive.coalescePartitions.enabled", "false")
set_conf_safe("spark.sql.shuffle.partitions", 512)

In [0]:
# -- GeoBrix tier selection (keep in sync with library.py) --
# option-1: lightweight tier (pure Python / PySpark, runs on Serverless) -- DEFAULT
from databricks.labs.gbx.pyrx import functions as rx          # <- rx.rst_* python bindings

# option-2: heavyweight tier (Scala JAR + GDAL init script on a classic x86 cluster)
# from databricks.labs.gbx.rasterx import functions as rx

rx.register(spark)                                            # <- invoke rst_* via spark sql

# -- register the light readers/writers (gtiff_gbx, shapefile_gbx, ...)
from databricks.labs.gbx.ds.register import register
register(spark)

In [0]:
# -- rebuild control: when True, force re-create of tables / re-download by feeding the
#    existing do_overwrite / do_append params and skip-guards below. Set per-notebook
#    (after %run ./config_nb) to fully re-exercise that notebook's logic.
FORCE_REBUILD = False

catalog_name = "geospatial_docs"
schema_name = "eo_series"

sql(f"""USE CATALOG {catalog_name}""")
sql(f"""CREATE DATABASE IF NOT EXISTS {schema_name}""")
sql(f"""USE DATABASE {schema_name}""")

print(f"... catalog: '{catalog_name}' (USE)")
print(f"... schema: '{schema_name}' (CREATE / USE)")

In [0]:
ETL_DIR = f"/Volumes/{catalog_name}/{schema_name}/data" # <- Volume ('data') must exist
EO_DIR = f"{ETL_DIR}/alaska"
dbutils.fs.mkdirs(EO_DIR)

os.environ['ETL_DIR'] = ETL_DIR
os.environ['EO_DIR'] = EO_DIR

print(f"... ETL_DIR: '{ETL_DIR}'")
print(f"... EO_DIR: '{EO_DIR}' (MKDIRS)")

In [0]:
# -- library.py
import library

In [0]:
%reload_ext autoreload
%autoreload 2
%reload_ext library

In [0]:
# -- 01. Viz Support --

In [0]:
def as_gdf(df, wkt_col = "wkt"):
  gs = gpd.GeoSeries.from_wkt(
    df
    .toPandas()[wkt_col], 
    crs=4326
  )
  pdf = df.drop(wkt_col).toPandas()
  pdf["geometry"] = gs
  gdf = gpd.GeoDataFrame(pdf, geometry="geometry")
  return gdf

In [0]:
def cells_as_gdf(df, cell_col = "cellid", extra_cols=[]):
  cols = [cell_col]
  cols.extend(extra_cols)
  return as_gdf(
    (
      df
        .select(*cols)
        .withColumn("wkt", DBF.h3_boundaryaswkt(cell_col))
    )
  )

In [0]:
# -- 02. Download STACs

In [0]:
@udf(returnType=IntegerType())
def file_size(file_path):
  """
  Return file_size or null.
  - must exist and be a file
  """
  import os

  if not file_path or not isinstance(file_path, (str, bytes, os.PathLike)):
        return None

  if os.path.exists(file_path) and os.path.isfile(file_path):
    return os.path.getsize(file_path)
  else:
    return None


@udf(returnType=StringType())
def timestamp_filename(dt):
  """
  Convert a passed timestamp to a filename friendly output.
  - return looks like 20230131-092030
  """
  from datetime import datetime

  if dt is None:
    return None
  return dt.strftime(library.FILENAME_TIMESTAMP_FORMAT)

def get_now_formatted():
  """
  Use for last update.
  - this is same as used in `timestamp_filename`
  """
  return datetime.now().strftime(library.FILENAME_TIMESTAMP_FORMAT)

In [0]:
# -- 03. Gridded EO Data

## Helper Functions

`finalize_tiled_band_tbl` and `gen_tessellate_tiled_band` are shared pipeline utilities used by notebooks 03 and 04. They use `rst_srid` (extract CRS identifier), `rst_boundingbox` (spatial extent as WKB), `rst_initnodata` (mask nodata pixels), and `rst_memsize` (in-memory footprint) for ingestion, and invoke `gbx_rst_h3_tessellate` via a SQL LATERAL to assign each tile to its overlapping H3 cells. See [RasterX Functions](https://databrickslabs.github.io/geobrix/docs/api/raster-functions) and [H3 Raster Tessellation](https://databrickslabs.github.io/geobrix/docs/api/h3-raster-tessellation).

In [0]:
def finalize_tiled_band_tbl(
  band_df, tbl_name, repartition_factor = 3,
  item_id_col="item_id", file_col="out_file_path", size_col="size", bbox_col="bbox",
  drop_cols=["timestamp", "h3_set", "asset", "item_properties", "item_collection", "item_bbox", "stac_version", 
        "last_update", "out_dir_fuse", "out_filename", "out_file_path", "out_file_sz", "is_out_file_valid"],
  do_try_open=False, do_display=True, do_overwrite=False, do_append=False
):
  """
  Finalize a tile table for a band dataframe.
  - tile_threshold: size in MB (default 32MB)
  - if you have some bad data, set do_try_open=True
  Returns the resulting dataframe (from the spark table generated).
  """
  _band_df = None
  try:
    if do_overwrite:
      sql(f"""drop table if exists {tbl_name}""")
    if not spark.catalog.tableExists(tbl_name) or do_append:
      _n = band_df.count()
      # one tile per task: parallelism + caps per-task Arrow/UDF memory (full
      # scenes are split upstream, but one-per-task keeps Serverless safe).
      _band_df = band_df.repartition(max(1, _n), F.col(file_col))
      band_cnt = _n
      print(f"... number of files to process? {band_cnt:,}")

      if do_try_open:
        _band_df = _band_df.filter(rx.rst_tryopen("tile"))
      (
        _band_df
          .withColumn(size_col, rx.rst_memsize("tile"))
          .withColumn("tile", rx.rst_initnodata("tile"))
          .withColumn(bbox_col, rx.rst_boundingbox("tile"))
          .withColumn("srid", rx.rst_srid("tile"))
        .drop(*drop_cols)
        .write
          .option("mergeSchema", "true")
          .mode("append")
          .saveAsTable(tbl_name)
      )
    sql(f"""ALTER TABLE {tbl_name} CLUSTER BY ({item_id_col})""")
    sql(f"""OPTIMIZE {tbl_name}""")

    tile_df = spark.table(tbl_name)
    if do_display:
      print(f"\n-- {tbl_name}")
      print(f"count? {tile_df.count():,}")
      tile_df.limit(10).display()
    return tile_df
  finally:
    pass  # nothing persisted (no cache on Serverless); explicit repartition handles parallelism

In [0]:
def gen_tessellate_tiled_band(
  tile_df, h3_res, tbl_name, h3_col="cellid",
  do_try_open=False, do_display=True, do_overwrite=False, do_append=False
):
  """
  Generate H3 tessellation table from a tiled band dataframe.
  - Expects 'tile' column to be present.
  - If you have bad tiffs, set do_try_open=True
  - MLJ: set max_records_per_file, defaults to 1000 (about control over the number of files created)
  Returns the resulting dataframe (from the spark table generated).
  """
  if do_overwrite:
      sql(f"""drop table if exists {tbl_name}""")
  if not spark.catalog.tableExists(tbl_name) or do_append:
    if do_try_open:
      _tile_df = tile_df.filter(rx.rst_tryopen("tile")).filter(F.isnotnull("tile"))
    else:
      _tile_df = tile_df.filter(F.isnotnull("tile"))
    
    # rst_h3_tessellate is a UDTF (SQL LATERAL table function) in the lightweight
    # tier, not a column function -- invoke it via LATERAL. It yields TILE_SCHEMA
    # columns (cellid, raster, metadata) per overlapping H3 cell; rebuild the `tile`
    # struct so downstream rst_* ops keep working. Explicit repartition (one tile per
    # task) = Serverless-safe parallelism (not AQE-coalesced) + caps per-task memory.
    _tile_df = _tile_df.withColumn("_repart_key", F.monotonically_increasing_id())
    _tile_df = _tile_df.repartition(max(1, _tile_df.count()), F.col("_repart_key"))
    _carry = [c for c in _tile_df.columns if c not in ("tile", "_repart_key")]   # keep the key out of output
    _carry_sel = "".join([f", s.`{c}`" for c in _carry])
    _tile_df.createOrReplaceTempView("_gtess_input")
    write_df = spark.sql(f"""
      SELECT
        t.cellid AS {h3_col},
        named_struct('cellid', t.cellid, 'raster', t.raster, 'metadata', t.metadata) AS tile
        {_carry_sel}
      FROM _gtess_input s,
      LATERAL gbx_rst_h3_tessellate(s.tile, {h3_res}) t
    """)
    (
      write_df.write
        .option("mergeSchema", "true")
        .mode("append")
      .saveAsTable(tbl_name)
    )
    sql (f"""ALTER TABLE {tbl_name} CLUSTER BY ({h3_col})""")
    sql(f"""OPTIMIZE {tbl_name}""")

  h3_df = spark.table(tbl_name)
  if do_display:
    print(f"\n-- {tbl_name}")
    print(f"count? {h3_df.count():,}")
    h3_df.limit(10).display()
  return h3_df